In [ ]:
%matplotlib notebook
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import cvxpy as cp
from IPython.display import HTML, display
import time

def make_grid(nx, ny, spacing=1.0):
    N = nx * ny
    pts = np.column_stack([np.repeat(np.arange(nx)*spacing, ny), 
                           np.tile(np.arange(ny)*spacing, nx), 
                           np.zeros(N)])
    idx = np.arange(N).reshape(nx, ny)
    
    e_main = np.vstack([np.column_stack([idx[:-1,:].ravel(), idx[1:,:].ravel()]),
                        np.column_stack([idx[:,:-1].ravel(), idx[:,1:].ravel()])])
    e_diag = np.vstack([np.column_stack([idx[:-1,:-1].ravel(), idx[1:,1:].ravel()]),
                        np.column_stack([idx[1:,:-1].ravel(), idx[:-1,1:].ravel()])])
    
    return pts, e_main, e_diag

class ClothSOCP:
    def __init__(self, N, all_edges, L_targets, weights, anchors,
                 lambda_inertia=200.0, gamma_gravity=1.0, dmax=0.5, s_slack=1e-4, solver=cp.CLARABEL):
        self.solver = solver
        self.X = cp.Variable((N, 3))
        self.s = cp.Variable(len(all_edges), nonneg=True)
        self.x_prev = cp.Parameter((N, 3))
        self.x_pred = cp.Parameter((N, 3))

        cons = [cp.norm(self.X[i] - self.X[j]) <= self.s[k] for k, (i, j) in enumerate(all_edges)]
        cons += [self.s <= L_targets + s_slack]
        cons += [self.X[i] == self.x_prev[i] for i in anchors]
        free_idx = [i for i in range(N) if i not in anchors]
        cons += [cp.abs(self.X[free_idx] - self.x_prev[free_idx]) <= dmax / np.sqrt(3.0)]

        inertia = lambda_inertia * cp.sum_squares(self.X - self.x_pred)
        gravity = -gamma_gravity * cp.sum(self.X[:, 2])
        penalty = cp.sum(cp.multiply(weights, cp.square(self.s - L_targets)))
        
        self.prob = cp.Problem(cp.Minimize(inertia + gravity + penalty), cons)
        self.warm_X, self.warm_s = None, None

    def solve(self, x_cur, v_cur, dt):
        self.x_prev.value = x_cur
        self.x_pred.value = x_cur + v_cur * dt + 0.5 * np.array([0, 0, -9.8]) * dt**2
        
        if self.warm_X is not None:
            self.X.value, self.s.value = self.warm_X, self.warm_s

        self.prob.solve(solver=self.solver, warm_start=True, verbose=False)
        self.warm_X, self.warm_s = self.X.value, self.s.value
        
        return self.prob, self.warm_X, self.warm_s

nx, ny, dt = 10, 10, 0.05
pts, e_main, e_diag = make_grid(nx, ny)
all_edges = np.vstack([e_main, e_diag])

L_tgt = np.concatenate([np.full(len(e_main), 1.0), np.full(len(e_diag), np.sqrt(2.0))])
weights = np.concatenate([np.full(len(e_main), 500.0), np.full(len(e_diag), 300.0)])

solver = ClothSOCP(nx*ny, all_edges, L_tgt, weights, anchors={0, nx*ny - ny}, solver=cp.CLARABEL)
x_cur, v_cur = pts.copy(), np.zeros_like(pts)

history, logic_times = [], []

for step in range(150):
    t0 = time.perf_counter()
    prob, Xopt, s_opt = solver.solve(x_cur, v_cur, dt)
    t1 = time.perf_counter()
    logic_times.append(t1 - t0)

    if Xopt is None:
        print(f"Solver failed at step {step}: {prob.status}")
        break

    if step % 10 == 0:
        max_edge = np.linalg.norm(Xopt[all_edges[:,0]] - Xopt[all_edges[:,1]], axis=1).max()
        avg_t = np.mean(np.abs(s_opt - L_tgt))
        max_disp = np.linalg.norm(Xopt - x_cur, axis=1).max()
        print(f"step {step:3d} | logic_time: {(t1-t0)*1000:.2f} ms")

    history.append(Xopt.copy())
    v_cur = (Xopt - x_cur) / dt
    x_cur = Xopt

fig = plt.figure(figsize=(6,5))
ax = fig.add_subplot(111, projection='3d')
ax.set(xlim=(-3, 7), ylim=(-3, 7), zlim=(-9, 1), xlabel='X', ylabel='Y', zlabel='Z')
ax.view_init(elev=30, azim=-60)

scat = ax.scatter(history[0][:,0], history[0][:,1], history[0][:,2], c='tab:blue', s=30)
lc = Line3DCollection(history[0][all_edges], colors='gray', lw=1)
ax.add_collection3d(lc)

def update_frame(k):
    scat._offsets3d = (history[k][:,0], history[k][:,1], history[k][:,2])
    lc.set_segments(history[k][all_edges])
    ax.set_title(f"t = {k}")
    return [scat, lc]

anim = animation.FuncAnimation(fig, update_frame, frames=len(history), interval=80, blit=False)
display(HTML(anim.to_jshtml()))